# Clean the raw transaction data

Set the project root so this notebook works whether Jupyter starts in the repository root or `notebooks/`. Import the shared paths and seed from `src/config.py`, then load the raw CSV without modifying it.

In [1]:
import os
import json
import random
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "config.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "config.py").is_file():
    raise FileNotFoundError("Could not locate src/config.py from the current directory")
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd

from src.config import ARTIFACTS, CLEAN_PATH, RAW_PATH, SEED

random.seed(SEED)
np.random.seed(SEED)

df = pd.read_csv(RAW_PATH)
print("Loaded raw dataset from:", RAW_PATH)
print("Raw shape:", df.shape)

Loaded raw dataset from: data/raw/creditcard.csv
Raw shape: (284807, 31)


## Validate the source data

Check required columns, numeric types, allowed class labels, and non-negative amounts before cleaning. Any failed check raises a clear `ValueError`.

In [2]:
required_columns = ["Time"] + ["V{}".format(i) for i in range(1, 29)] + ["Amount", "Class"]
validation_errors = []

missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    validation_errors.append("missing required columns: {}".format(missing_columns))

non_numeric_columns = [
    column for column in df.columns
    if not pd.api.types.is_numeric_dtype(df[column])
]
if non_numeric_columns:
    validation_errors.append("columns must all be numeric; non-numeric: {}".format(non_numeric_columns))

if "Class" in df.columns:
    invalid_classes = sorted(set(df["Class"].dropna().unique()) - {0, 1})
    if invalid_classes:
        validation_errors.append("Class contains values outside {0, 1}: {}".format(invalid_classes))

if "Amount" in df.columns and (df["Amount"].dropna() < 0).any():
    validation_errors.append("Amount contains negative values")

if validation_errors:
    raise ValueError("Raw data validation failed: " + "; ".join(validation_errors))

print("Validation passed: required columns are present, all columns are numeric, Class is in {0, 1}, and Amount is non-negative.")

Validation passed: required columns are present, all columns are numeric, Class is in {0, 1}, and Amount is non-negative.


## Remove incomplete and duplicate rows

Record the original data-quality counts, remove rows with missing values and then exact duplicate rows, and report the number of rows removed at each step. Keep every original column, sort by `Time`, and reset the index.

In [3]:
before_rows = int(len(df))
missing_value_count = int(df.isna().sum().sum())
rows_with_missing_values = int(df.isna().any(axis=1).sum())
df_clean = df.dropna().copy()
missing_rows_removed = before_rows - int(len(df_clean))
duplicate_rows_removed = int(df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates()

print("Missing values found:", missing_value_count)
print("Rows containing missing values removed:", missing_rows_removed)
print("Exact duplicate rows removed:", duplicate_rows_removed)

df_clean = df_clean.sort_values("Time", ascending=True).reset_index(drop=True)

assert not df_clean.isna().any().any(), "Cleaned data still contains missing values"
assert not df_clean.duplicated().any(), "Cleaned data still contains duplicate rows"
assert df_clean["Time"].is_monotonic_increasing, "Cleaned data is not sorted by Time"
assert set(df_clean["Class"].unique()).issubset({0, 1}), "Cleaned data contains an invalid Class value"
assert list(df_clean.columns) == list(df.columns), "Cleaned data columns differ from the source columns"

print("All cleaning assertions passed; original columns are retained.")

Missing values found: 0
Rows containing missing values removed: 0
Exact duplicate rows removed: 1081


All cleaning assertions passed; original columns are retained.


## Compare and save

Compare rows, duplicate rows, null cells, fraud counts, and fraud rates before and after cleaning. Write the cleaned CSV to the configured path and save the comparison and removal counts as JSON in the configured artifacts directory.

In [4]:
def dataset_summary(frame):
    fraud_count = int((frame["Class"] == 1).sum())
    return {
        "rows": int(len(frame)),
        "duplicates": int(frame.duplicated().sum()),
        "nulls": int(frame.isna().sum().sum()),
        "frauds": fraud_count,
        "fraud_rate": float(fraud_count / len(frame)) if len(frame) else 0.0,
    }

before_summary = dataset_summary(df)
after_summary = dataset_summary(df_clean)
comparison = pd.DataFrame([before_summary, after_summary], index=["before", "after"])
comparison["fraud_rate"] = comparison["fraud_rate"] * 100
comparison = comparison.rename(columns={"fraud_rate": "fraud_rate_percent"})
display(comparison)

clean_path = Path(CLEAN_PATH)
clean_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(clean_path, index=False)

clean_report = {
    "raw_path": RAW_PATH,
    "clean_path": CLEAN_PATH,
    "removed": {
        "missing_values": missing_value_count,
        "rows_with_missing_values": missing_rows_removed,
        "duplicate_rows": duplicate_rows_removed,
    },
    "before": before_summary,
    "after": after_summary,
}
clean_report_path = Path(ARTIFACTS) / "clean_report.json"
clean_report_path.write_text(
    json.dumps(clean_report, indent=2) + "\n",
    encoding="utf-8",
)
print("Saved cleaned dataset to:", clean_path)
print("Saved cleaning report to:", clean_report_path)
print("Cleaned shape:", df_clean.shape)

,rows,duplicates,nulls,frauds,fraud_rate_percent
before,284807,1081,0,492,0.172749
after,283726,0,0,473,0.166710


Saved cleaned dataset to: data/processed/creditcard_clean.csv
Saved cleaning report to: artifacts/clean_report.json
Cleaned shape: (283726, 31)
